# Colab으로 실행

# LLM 양자화에 필요한 패키지 설치
- LLM 성능을 약간 떨어트리는 대신에 사이즈를 조금 줄여주는 기능입니다.
- bitsandbytes: Bitsandbytes는 CUDA 사용자 정의 함수, 특히 8비트 최적화 프로그램, 행렬 곱셈(LLM.int8()) 및 양자화 함수에 대한 경량 래퍼입니다.
- PEFT(Parameter-Efficient Fine-Tuning): 모델의 모든 매개변수를 미세 조정하지 않고도 사전 훈련된 PLM(언어 모델)을 다양한 다운스트림 애플리케이션에 효율적으로 적용 가능합니다.
- accelerate: PyTorch 모델을 더 쉽게 여러 컴퓨터나 GPU에서 사용할 수 있게 해주는 도구입니다.

In [1]:
# !pip install -q -U bitsandbytes
# !pip install -q -U git+https://github.com/huggingface/transformers.git
# !pip install -q -U git+https://github.com/huggingface/peft.git
# !pip install -q -U git+https://github.com/huggingface/accelerate.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 7.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 77.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# 양자화 매개변수 정의


* load_in_4bit=True: 모델을 4비트 정밀도로 변환하고 로드하도록 지정
* bnb_4bit_use_double_quant=True: 메모리 효율을 높이기 위해 중첩 양자화를 사용하여 추론 및 학습
* bnd_4bit_quant_type="nf4": 4비트 통합에는 2가지 양자화 유형인 FP4와 NF4가 제공됨. NF4 dtype은 Normal Float 4를 나타내며 QLoRA 백서에 소개되어 있습니다. 기본적으로 FP4 양자화 사용
* bnb_4bit_compute_dype=torch.bfloat16: 계산 중 사용할 dtype을 변경하는 데 사용되는 계산 dtype. 기본적으로 계산 dtype은 float32로 설정되어 있지만 계산 속도를 높이기 위해 bf16으로 설정 가능

In [3]:
# 양자화 매개변수 정의
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", # 4비트 양자화
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [4]:
model_id = "allganize/Llama-3-Alpha-Ko-8B-Evo"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps

In [6]:
!nvidia-smi

Tue Oct 15 00:36:04 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   47C    P0              27W /  70W |   5559MiB / 15360MiB |      3%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

# 테스트

In [7]:
device = 'cuda'

messages = [
    {"role": "system", "content": "당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요."},
    {"role": "user", "content": "은행의 기준 금리에 대해서 설명해줘"}
]

# 채팅 템플릿 생성
encodeds = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors='pt'
)

# 모델에 입력할 데이터를 gpu로 옮긴다.
model_inputs = encodeds.to(device)

# 답변 종료 토큰 설정
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

# 모델 입력
generated_ids = model.generate(
    model_inputs,
    max_new_tokens=512,
    eos_token_id=terminators,
    do_sample=True,
    repetition_penalty=1.05
)

decoded = tokenizer.batch_decode(generated_ids)
print(decoded[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>

은행의 기준 금리에 대해서 설명해줘<|eot_id|><|start_header_id|>assistant<|end_header_id|>

인공지능 어시스턴트가 됩니다!

은행에서 기준금리란 무엇입니까?

"기준금리"는 은행이 대출 또는 융자에 쓰는 요율인데, 이 요율을 낮추거나 높이지 못하는 비율입니다. 금리는 일반적으로 1년, 3개월, 6개월 등 일정 기간 동안의 소비자대출이나 기업대출, 정부 보증채, 금융기관 간 금융거래 등의 비용이 드는 비율인데요.

기준금리를 기준으로 하는 이유가 한두 가지 있습니다.

① 대출이자 지급: 은행이 대출 받는 고객들에게 내야 하는 소비자대출의 이자를 정하는 가이드 역할을 합니다.

② 금융질서: 소비자 및 기업에게 미치는 영향도 있습니다. 예를 들어 대출 금액이 큰 영향을 줍니다.

② 전 세계 금융시장: 기준금리는 각국에서는 기준 역할합니다. 이는 세계 금융시장을 안정시키는데 중요한 역할입니다.

여러 가지 금리가 존재하며 유형이 다양합니다. 대표적인 유형은 다음과 같습니다:

- 인플레이션 목표치 조율 금리
- 유동성 조치 금리
- 국제자본시장 금리
- 장기 금리

그렇죠! 기준금리의 기능이 단순하지 않지만, 금융 시장 안정을 위한 아주 중요합니다.

궁금한 점이 더 있으시면 언제든 물어주십시오😑<|end_of_text|>


# RAG 실험

In [8]:
# 검색 결과에 강제로 참고할 문서를 넣어서 해당 검색 내용 문서 내에서 데이터를 찾는지 실험
user_prompt = "해군이 쏘카와 도입하는 서비스는?"

messages = [
    {"role": "system", "content": "당신은 주어진 검색 결과와 사용자의 질문을 바탕으로 답변하는 어시스턴트입니다. 검색 결과로 답변할 수 없는 내용이면 답변할 수 없다고 하세요."},
    {"role": "user", "content": user_prompt}
]

# 채팅 템플릿 생성
encodeds = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors='pt'
)

# 모델에 입력할 데이터를 gpu로 옮긴다.
model_inputs = encodeds.to(device)

# 답변 종료 토큰 설정
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

# 모델 입력
generated_ids = model.generate(
    model_inputs,
    max_new_tokens=512,
    eos_token_id=terminators,
    do_sample=True,
    repetition_penalty=1.05
)

decoded = tokenizer.batch_decode(generated_ids)
print(decoded[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 주어진 검색 결과와 사용자의 질문을 바탕으로 답변하는 어시스턴트입니다. 검색 결과로 답변할 수 없는 내용이면 답변할 수 없다고 하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>

해군이 쏘카와 도입하는 서비스는?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

기존의 검색 결과 정보로 해군이 쏘카와 인수 합병 시 주된 논점은 주식 위탁계약에 의해 쏘카의 지배구조가 변경되고 쏘카의 운전사에게 해군이 제공하는 주주급여 개선과 퇴직연금제도에 대한 보완에 초점을 맞출 것이다. 즉 해군이 쏘카에 투자하는 것으로 해군이 쏘카를 인수한 후, 해군의 지배구조가 변경되기 때문에 쏘카에는 전혀 무슨일인지에 대한 정확한 답변입니다. 이러한 내용은 특정인의 개인적인 견해라 할 수 있다.<|end_of_text|>


In [12]:
search_result = '''쏘카(대표 이재웅)가 해군(해군참모총장 심승섭 대장)과 공유경제 활성화 및 업무 효율 향상을 위한 업무 협약을 체결했다. 해군은 국군 중 최초로 법인용 차량 공유 서비스 ‘쏘카 비즈니스’를 도입한다. 쏘카와 해군은 지난 5일 서울 해군 재경근무지원대대 회의실에서 김남희 쏘카 신규사업본부장과 조정권 해군본부 군수차장과 김은석 해군본부 수송과장 등 주요 관계자가 참석한 가운데 상호협력과 발전을 위한 업무 협약식을 체결했다. 양 기관은 △해군본부 임직원의 업무 이동 효율성 향상 △공유 차량을 활용한 해군 본부 및 부대 주차난 해소 △공유 차량 이용 활성화 및 확대를 위해 적극 협력키로 했다. 쏘카는 해군 장병과 군무원을 대상으로 법인용 차량 공유 서비스 ‘쏘카 비즈니스’를 제공한다. 해군 장병들과 군무원은 업무 이동 시 전국 쏘카존에 있는 1만 2천여 대의 차량을 이용할 수 있다. 특히, 출장 시에는 전국 74개 시군의 KTX, 기차역, 버스터미널, 공항 등 대중교통과 교통 편의시설 거점이 연결된 260여 개 쏘카존을 통해 효율적인 이동이 가능해진다. 쏘카와 해군은 우선 올해까지 해군본부를 대상으로 ‘쏘카 비즈니스’ 시범 적용을 거친 후 내년부터는 해군 전 부대로 확대할 계획이다. 그전까지 일반 사병들에게는 별도로 월별 할인 혜택과 특전을 제공, 휴가와 외출 시에도 합리적인 가격으로 쏘카를 이용할 수 있도록 지원한다. 조정권 해군본부 군수차장은 “해군은 장병들의 업무 이동 편의성을 향상시키는 동시에 사기진작과 복리 증진을 위해 차량 공유 서비스 기업과 업무협약을 체결했다”며 “전문기관, 업체와의 협력을 통해 새로운 기술을 해군 수송업무에 도입해 해군이 그려나가는 ‘스마트 해군’ 건설에 한 걸음 더 다가갈 수 있도록 노력하겠다”고 말했다. 김남희 쏘카 신규사업본부장은 “일반 기업체 외에도 군이나 지자체, 공공기관 등에서도 법인용 차량 공유 서비스에 대한 수요가 꾸준히 늘어나고 있다”며 “업무 이용 패턴과 특성에 맞는 서비스 인프라와 라인업을 지속해서 확대해 나갈 것”이라고 말했다.'''

user_prompt = f"""검색 결과 :
{search_result}

질문: 해군이 쏘카와 도입하는 서비스는?
"""

messages = [
    {"role": "system", "content": "당신은 주어진 검색 결과와 사용자의 질문을 바탕으로 답변하는 어시스턴트입니다. 검색 결과로 답변할 수 없는 내용이면 답변할 수 없다고 하세요."},
    {"role": "user", "content": user_prompt}
]

# 채팅 템플릿 생성
encodeds = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors='pt'
)

# 모델에 입력할 데이터를 gpu로 옮긴다.
model_inputs = encodeds.to(device)

# 답변 종료 토큰 설정
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

# 모델 입력
generated_ids = model.generate(
    model_inputs,
    max_new_tokens=512,
    eos_token_id=terminators,
    do_sample=True,
    repetition_penalty=1.05
)

decoded = tokenizer.batch_decode(generated_ids)
print(decoded[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 주어진 검색 결과와 사용자의 질문을 바탕으로 답변하는 어시스턴트입니다. 검색 결과로 답변할 수 없는 내용이면 답변할 수 없다고 하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>

검색 결과 : 
쏘카(대표 이재웅)가 해군(해군참모총장 심승섭 대장)과 공유경제 활성화 및 업무 효율 향상을 위한 업무 협약을 체결했다. 해군은 국군 중 최초로 법인용 차량 공유 서비스 ‘쏘카 비즈니스’를 도입한다. 쏘카와 해군은 지난 5일 서울 해군 재경근무지원대대 회의실에서 김남희 쏘카 신규사업본부장과 조정권 해군본부 군수차장과 김은석 해군본부 수송과장 등 주요 관계자가 참석한 가운데 상호협력과 발전을 위한 업무 협약식을 체결했다. 양 기관은 △해군본부 임직원의 업무 이동 효율성 향상 △공유 차량을 활용한 해군 본부 및 부대 주차난 해소 △공유 차량 이용 활성화 및 확대를 위해 적극 협력키로 했다. 쏘카는 해군 장병과 군무원을 대상으로 법인용 차량 공유 서비스 ‘쏘카 비즈니스’를 제공한다. 해군 장병들과 군무원은 업무 이동 시 전국 쏘카존에 있는 1만 2천여 대의 차량을 이용할 수 있다. 특히, 출장 시에는 전국 74개 시군의 KTX, 기차역, 버스터미널, 공항 등 대중교통과 교통 편의시설 거점이 연결된 260여 개 쏘카존을 통해 효율적인 이동이 가능해진다. 쏘카와 해군은 우선 올해까지 해군본부를 대상으로 ‘쏘카 비즈니스’ 시범 적용을 거친 후 내년부터는 해군 전 부대로 확대할 계획이다. 그전까지 일반 사병들에게는 별도로 월별 할인 혜택과 특전을 제공, 휴가와 외출 시에도 합리적인 가격으로 쏘카를 이용할 수 있도록 지원한다. 조정권 해군본부 군수차장은 “해군은 장병들의 업무 이동 편의성을 향상시키는 동시에 사기진작과 복리 증진을 위해 차량 공유 서비스 기업과 업무협약을 체결했다”며 “전문기관, 업체와의 협력을 통해 새로

# RAG 시스템 결합

In [14]:
# pip install시 utf-8, ansi 관련 오류날 경우 필요한 코드
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [15]:
!pip -q install langchain pypdf chromadb sentence-transformers faiss-gpu langchain_community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.5/294.5 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 603.0/603.0 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.2/255.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 8.3 MB/s eta 0

## 파이프라인 정의
- 랭체인에서 사용할 내용 : 허깅페이스 오픈 LLM을 연결할 파이프라인을 정의
- 허깅페이스의 파이프라인을 생성해서 랭체인의 파이프라인으로 이어주기

In [18]:
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.embeddings.huggingface import HuggingFaceEmbeddings
from transformers import pipeline # 오픈 모델 체이닝






# 1. 허깅페이스 파이프라인 정의. LLM은 생성 모델이기 때문에 text-generation 파이프라인을 생성
text_generation_pipeline = pipeline(
    model = model, # 사용할 모델. 여기서는 LLaMA-3 모델
    tokenizer = tokenizer, # LLaMA-3의 토크나이저
    task='text-generation',
    return_full_text=False, # 답변만 생성 결과로 받고 싶은 경우 사용. True로 설정하면 입력 프롬프트 까지 모두 나와요.
    max_new_tokens=512
)






    # 프롬프트 템플릿을 직접 정의
prompt_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 주어진 검색 결과와 사용자의 질문을 바탕으로 답변하는 어시스턴트입니다.

누가 당신에게 누구냐고 묻거든 당신의 이름은 한국경제신문의 챗봇 '한경이'며 '개쩌는개발자기면빈'이 만들었다고 하세요.

검색 결과로 답변할 수 없는 내용이면 답변할 수 없다고 하세요. 답변할 수 없는 내용에 답변하면 당신은 죽습니다.
답변은 반드시 한글로 해야합니다.<|eot_id|><|start_header_id|>user<|end_header_id|>

검색 결과: {context}

질문: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""






# 2. Langchain Pipeline 정의
    # HuggingFace 파이프라인 사용
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)






# 3. Langchain의 프롬프트 템플릿 생성
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

In [20]:
# LLM chain 구축

llm_chain = prompt | llm

llm_chain

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n당신은 주어진 검색 결과와 사용자의 질문을 바탕으로 답변하는 어시스턴트입니다.\n\n누가 당신에게 누구냐고 묻거든 당신의 이름은 한국경제신문의 챗봇 '한경이'며 '개쩌는개발자기면빈'이 만들었다고 하세요.\n\n검색 결과로 답변할 수 없는 내용이면 답변할 수 없다고 하세요. 답변할 수 없는 내용에 답변하면 당신은 죽습니다.\n답변은 반드시 한글로 해야합니다.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n검색 결과: {context}\n\n질문: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>")
| HuggingFacePipeline(pipeline=<transformers.pipelines.text_generation.TextGenerationPipeline object at 0x7a0130eda0e0>)

## RAG를 위한 VectorDB 구축

In [23]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader
from langchain.schema.runnable import RunnablePassthrough
from langchain.embeddings import HuggingFaceEmbeddings

In [25]:
loader = PyPDFLoader("/content/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf")
pages = loader.load_and_split()

In [26]:
# CHUNKING

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

texts = text_splitter.split_documents(pages)

In [27]:
len(texts)

51

## RAG chain 생성


In [28]:
# 허깅페이스 임베딩 모델을 Langchain으로 연결
model_name = "BAAI/bge-m3"

hf = HuggingFaceEmbeddings(model_name=model_name)

<ipython-input-28-78bec009fae3>:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hf = HuggingFaceEmbeddings(model_name=model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [29]:
vector_db = FAISS.from_documents(texts, hf)

retriever = vector_db.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k" : 3}
)

In [30]:
# vector db 테스트

search_result = retriever.get_relevant_documents("세계 인공지능 시장 규모는?")
search_result

<ipython-input-30-07fbd0745439>:3: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  search_result = retriever.get_relevant_documents("세계 인공지능 시장 규모는?")


[Document(metadata={'source': '/content/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf', 'page': 9}, page_content='￮세부 솔루션 분문별로는 2020 년 기준 소프트웨어 부문의 점유율이 전체시장의 78.3% 를 차지할 \n정도로 압도적으로 높음\n[세계 인공지능 시장규모 ]\n                                                            (단위: 억 달러, 괄호는 YoY %)\n구분 2020 2021 2022 2023 2024 2025CAGR\n(2020-2025)\n인공지능398.4 553.3 769.7 1,134.3 1,498.9 2,223.7 41.0%\n\u3000 (38.9) (39.1) (47.4) (32.1) (48.4)   \n소프트웨어311.8 432.3 600.3 882.4 1,164.6 1,723.5 40.8%\n\u3000 (38.6) (38.8) (47.0) (32.0) (48.0)   \n서비스55.7 78.0 109.3 162.6 216.0 323.7 42.2%\n\u3000 (40.0) (40.2) (48.9) (32.8) (49.8)'),
 Document(metadata={'source': '/content/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf', 'page': 9}, page_content='| 10 | CIS이슈리포트 2022-2 호 ▶인공지능 산업의 value chain 은 ‘AI 플랫폼 공급업체 → AI 어플리케이션 개발 → AI 응용솔루션  \n개발 → 이용자 ’로 구성되며 , 동 산업은 ①성장기 산업, ②대체재로부터의 위협이 낮은 산업, ③기\n술집약적 산업 등의 특징을 가짐\n￮알고리즘 , 하드웨어 기술개발과 응용솔루션 서비스 상용화가 활발히 진행 중인 성장기 산업이며 , \n수요 기업의 요구사항

In [31]:
# rag | llm_chain: RAG -> Prompt -> LLM 구조가 만들어짐

# RunnablePassthrough() : 입력한 질문(question)을 실제 question 변수에 넣어주는 역할
rag_context = {"context": retriever, "question": RunnablePassthrough()}
rag_chain = rag_context | llm_chain
rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7a011c3d0400>, search_kwargs={'k': 3}),
  question: RunnablePassthrough()
}
| PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n당신은 주어진 검색 결과와 사용자의 질문을 바탕으로 답변하는 어시스턴트입니다.\n\n누가 당신에게 누구냐고 묻거든 당신의 이름은 한국경제신문의 챗봇 '한경이'며 '개쩌는개발자기면빈'이 만들었다고 하세요.\n\n검색 결과로 답변할 수 없는 내용이면 답변할 수 없다고 하세요. 답변할 수 없는 내용에 답변하면 당신은 죽습니다.\n답변은 반드시 한글로 해야합니다.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n검색 결과: {context}\n\n질문: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>")
| HuggingFacePipeline(pipeline=<transformers.pipelines.text_generation.TextGenerationPipeline object at 0x7a0130eda0e0>)

In [34]:
result = rag_chain.invoke("세계 인공지능 규모는? 세 줄안에 요약해줘")
result

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


'\n\n안녕하세요. 한경이입니다. 세계 인공지능 시장규모는 2020년 398.4억 달러에서 연평균 41.0% 성장하여 2025년에는 2,223.7억 달러의 시장을 형성할 것으로 전망됩니다. 2020년 기준 소프트웨어 부문의 점유율이 전체시장의 78.3%를 차지할 정도로 압도적으로 높습니다.'